<a href="https://colab.research.google.com/github/NicoLatina/proyecto-logistica/blob/main/02_data_quality_and_cleaning_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/NicoLatina/proyecto-logistica.git

Cloning into 'proyecto-logistica'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 16 (delta 2), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 146.72 KiB | 9.78 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [2]:
import pandas as pd

df = pd.read_csv("/content/proyecto-logistica/data/raw/logistics_dataset.csv")

In [5]:
df.isna().sum() #Analizo nulos

,0
item_id,0
category,0
stock_level,0
reorder_point,0
reorder_frequency_days,0
lead_time_days,0
daily_demand,0
demand_std_dev,0
item_popularity_score,0
storage_location_id,0


In [10]:
df.duplicated().sum() #Analizo duplicados

np.int64(0)

In [11]:
df.duplicated(subset=["item_id"]).sum() #Analizo duplicados

np.int64(0)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3204 entries, 0 to 3203
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   item_id                    3204 non-null   object 
 1   category                   3204 non-null   object 
 2   stock_level                3204 non-null   int64  
 3   reorder_point              3204 non-null   int64  
 4   reorder_frequency_days     3204 non-null   int64  
 5   lead_time_days             3204 non-null   int64  
 6   daily_demand               3204 non-null   float64
 7   demand_std_dev             3204 non-null   float64
 8   item_popularity_score      3204 non-null   float64
 9   storage_location_id        3204 non-null   object 
 10  zone                       3204 non-null   object 
 11  picking_time_seconds       3204 non-null   int64  
 12  handling_cost_per_unit     3204 non-null   float64
 13  unit_price                 3204 non-null   float

In [18]:
df["last_restock_date"] = pd.to_datetime(df["last_restock_date"]) #Convierto variable

In [25]:
df["last_restock_date"].max()

Timestamp('2024-12-30 00:00:00')

In [26]:
df["last_restock_date"].min()

Timestamp('2024-01-01 00:00:00')

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3204 entries, 0 to 3203
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   item_id                    3204 non-null   object        
 1   category                   3204 non-null   object        
 2   stock_level                3204 non-null   int64         
 3   reorder_point              3204 non-null   int64         
 4   reorder_frequency_days     3204 non-null   int64         
 5   lead_time_days             3204 non-null   int64         
 6   daily_demand               3204 non-null   float64       
 7   demand_std_dev             3204 non-null   float64       
 8   item_popularity_score      3204 non-null   float64       
 9   storage_location_id        3204 non-null   object        
 10  zone                       3204 non-null   object        
 11  picking_time_seconds       3204 non-null   int64         
 12  handli

Analizo que no haya datos inconsistentes como stock negativo o ratios fuera de la escala 0 a 1

In [30]:
inventory_cols = [
    "stock_level",
    "reorder_point",
    "reorder_frequency_days",
    "lead_time_days"
]

(df[inventory_cols] < 0).sum()

,0
stock_level,0
reorder_point,0
reorder_frequency_days,0
lead_time_days,0


In [31]:
demand_cols = [
    "daily_demand",
    "demand_std_dev",
    "forecasted_demand_next_7d"
]

(df[demand_cols] < 0).sum()

,0
daily_demand,0
demand_std_dev,0
forecasted_demand_next_7d,0


In [32]:
operation_cols = [
    "picking_time_seconds",
    "handling_cost_per_unit",
    "holding_cost_per_unit_day",
    "unit_price"
]

(df[operation_cols] < 0).sum()

,0
picking_time_seconds,0
handling_cost_per_unit,0
holding_cost_per_unit_day,0
unit_price,0


In [33]:
order_cols = [
    "stockout_count_last_month",
    "total_orders_last_month"
]

(df[order_cols] < 0).sum()

,0
stockout_count_last_month,0
total_orders_last_month,0


In [34]:
ratio_cols = [
    "order_fulfillment_rate",
    "item_popularity_score",
    "layout_efficiency_score"
]

((df[ratio_cols] < 0) | (df[ratio_cols] > 1)).sum()

,0
order_fulfillment_rate,0
item_popularity_score,0
layout_efficiency_score,0


Tabla de rangos

In [37]:
df_num = df.select_dtypes(include="number").columns

rangos = pd.DataFrame({
    "min": df[df_num].min(),
    "max": df[df_num].max(),
    "promedio": df[df_num].mean()
})

rangos

,min,max,promedio
stock_level,20.000,499.000,263.491573
reorder_point,10.000,99.000,54.759363
reorder_frequency_days,3.000,14.000,8.507803
lead_time_days,2.000,9.000,5.578340
daily_demand,1.010,49.980,25.435868
demand_std_dev,0.500,10.000,5.260078
item_popularity_score,0.100,1.000,0.542325
picking_time_seconds,10.000,179.000,95.606429
handling_cost_per_unit,0.500,5.000,2.777116
unit_price,10.220,200.000,105.887575


Reviso Outliers con metodo IQR

In [54]:
def detectar_outliers_iqr(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    iqr = Q3 - Q1

    limite_inferior = Q1 - 1.5 * iqr
    limite_superior = Q3 + 1.5 * iqr

    outliers = df[
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    ]

    return {
        "variable": columna,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": iqr,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "cantidad_outliers": len(outliers)
    }

In [55]:
variables_outliers = [
    "stock_level",
    "daily_demand",
    "picking_time_seconds",
    "unit_price",
    "turnover_ratio",
    "KPI_score"
]

In [56]:
resultados = []

for variable in variables_outliers:
    resultados.append(detectar_outliers_iqr(df, variable))

resumen_outliers = pd.DataFrame(resultados)
resumen_outliers

,variable,Q1,Q3,IQR,limite_inferior,limite_superior,cantidad_outliers
0,stock_level,144.000,386.00000,242.00000,-219.000000,749.000000,0
1,daily_demand,13.535,37.41250,23.87750,-22.281250,73.228750,0
2,picking_time_seconds,53.000,138.00000,85.00000,-74.500000,265.500000,0
3,unit_price,59.760,152.41000,92.65000,-79.215000,291.385000,0
4,turnover_ratio,4.590,11.69250,7.10250,-6.063750,22.346250,0
5,KPI_score,0.527,0.67725,0.15025,0.301625,0.902625,6


In [58]:
columna = 'KPI_score'
Q1 = df[columna].quantile(0.25)
Q3 = df[columna].quantile(0.75)
iqr = Q3 - Q1

limite_inferior = Q1 - 1.5 * iqr
limite_superior = Q3 + 1.5 * iqr

outliers_kpi = df[
    (df[columna] < limite_inferior) |
    (df[columna] > limite_superior)
]
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)
print("Cantidad de outliers:", len(outliers_kpi))
display(outliers_kpi)

Límite inferior: 0.30162500000000003
Límite superior: 0.902625
Cantidad de outliers: 6


,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,storage_location_id,...,unit_price,holding_cost_per_unit_day,stockout_count_last_month,order_fulfillment_rate,total_orders_last_month,turnover_ratio,layout_efficiency_score,last_restock_date,forecasted_demand_next_7d,KPI_score
63,ITM10063,Apparel,207,43,13,3,41.65,9.44,0.44,L81,...,17.38,1.51,9,0.70,327,1.18,0.24,2024-05-31,130.90,0.273
517,ITM10517,Automotive,78,23,5,6,1.91,3.06,0.44,L90,...,132.84,1.98,9,0.73,486,1.55,0.38,2024-12-09,176.39,0.265
558,ITM10558,Electronics,235,47,11,9,38.83,8.42,0.73,L56,...,72.53,0.19,0,0.99,458,13.46,0.83,2024-04-05,49.77,0.924
936,ITM10936,Electronics,471,47,7,7,3.57,1.77,0.34,L93,...,176.19,1.81,9,0.75,265,1.80,0.23,2024-09-24,17.64,0.259
2248,ITM12248,Electronics,389,64,13,3,12.55,4.14,0.79,L46,...,151.96,1.94,9,0.82,924,3.49,0.21,2024-11-22,275.28,0.279
3152,ITM13152,Groceries,45,49,7,2,14.52,1.86,0.41,L53,...,198.43,0.14,0,0.99,588,14.57,0.79,2024-04-10,37.40,0.936


In [59]:
columnas_revision = [
    "item_id",
    "category",
    "KPI_score",
    "stockout_count_last_month",
    "order_fulfillment_rate",
    "turnover_ratio",
    "layout_efficiency_score",
    "holding_cost_per_unit_day",
    "picking_time_seconds"
]

display(
    outliers_kpi[columnas_revision]
    .sort_values("KPI_score")
)

,item_id,category,KPI_score,stockout_count_last_month,order_fulfillment_rate,turnover_ratio,layout_efficiency_score,holding_cost_per_unit_day,picking_time_seconds
936,ITM10936,Electronics,0.259,9,0.75,1.80,0.23,1.81,25
517,ITM10517,Automotive,0.265,9,0.73,1.55,0.38,1.98,109
63,ITM10063,Apparel,0.273,9,0.70,1.18,0.24,1.51,165
2248,ITM12248,Electronics,0.279,9,0.82,3.49,0.21,1.94,90
558,ITM10558,Electronics,0.924,0,0.99,13.46,0.83,0.19,154
3152,ITM13152,Groceries,0.936,0,0.99,14.57,0.79,0.14,67


Los valores identificados corresponden a outliers estadísticos, pero no presentan indicios de ser errores de datos. Se observa coherencia entre el KPI_score y las variables asociadas al desempeño operativo, por lo que se decide conservar estos registros para las siguientes etapas del análisis.

#Data Quality Summary

Se evaluaron valores faltantes, registros duplicados, unicidad de los identificadores, consistencia de variables categóricas, tipos de datos, rangos numéricos y posibles valores atípicos.

El dataset presentó una buena calidad general, sin observarse valores faltantes ni registros duplicados. Los identificadores item_id resultaron únicos para cada SKU.

La principal transformación necesaria fue la conversión de last_restock_date desde texto hacia un formato de fecha adecuado.

Los valores extremos detectados fueron conservados siempre que se consideraran operacionalmente plausibles, dado que podrían representar situaciones relevantes como exceso de inventario, alta demanda o tiempos elevados de picking.